# TSFabrics pilot: Stage 2 (GPU)

This notebook only **launches** `scripts/run_stage2.py`. All logic lives in the repository.

**Inputs to attach (Add Input):**
1. The TSFabrics dataset (the one used for Stage 1).
2. This repository at the Stage 2 commit, uploaded as a Kaggle dataset (zip of the GitHub repo).

**Settings:** Accelerator **GPU** (one GPU; pick T4 or P100 and use the same type again for Stage 3). **Internet ON**
(pinned packages and the torchvision weights are downloaded).

**What Stage 2 reads** (enforced by `src/tsfpilot/access.py`): the 300 bank frames per fold, validation frames of
`T1_S164_I199_1` (1 to 1,212), and full training-side scenarios for phase correlation only. No test-scenario frame
is scored. The run log prints progress and acceptance results only.

**Run:** use **Save Version, Save & Run All (Commit)** so the run survives a closed browser. Expected time: about
1 to 2 hours. A smoke run (a few minutes) goes first; the full run starts only if the smoke run completes.

**PASS:** the last cell prints `STAGE 2 RESULT: PASS`.

**Send back:** the last cell's printout and `stage2_record.zip` from the Output tab. Keep the whole
`stage2_out/` folder (coreset vectors and validation maps) as a Kaggle dataset for Stage 3.

In [ ]:
import os, sys, json, shutil, subprocess, hashlib, zipfile

EXPECTED_PREREG_SHA256 = "182df443b29e8b5ee1922de61219871c1a6e052f538fd8ef1c4b4466db4a70dd"
SKIP_DIRS = {"IMAGE", "ANNOTATION", "CSV", ".git", "__pycache__"}

def bounded_walk(top, max_depth=5):
    """os.walk that never descends into dataset image/annotation/label folders."""
    base_depth = top.rstrip(os.sep).count(os.sep)
    for base, dirs, files in os.walk(top):
        dirs[:] = sorted(d for d in dirs if d not in SKIP_DIRS)
        if base.count(os.sep) - base_depth >= max_depth:
            dirs[:] = []
        yield base, dirs, files

# 1. Locate the repository copy (must contain PREREGISTRATION.md, src/tsfpilot and the Stage 2 runner)
repos = [b for b, d, f in bounded_walk("/kaggle/input")
         if "PREREGISTRATION.md" in f and os.path.isfile(os.path.join(b, "scripts", "run_stage2.py"))]
assert len(repos) == 1, f"expected exactly one Stage 2 repository copy under /kaggle/input, found: {repos}"
WORK = "/kaggle/working/repo"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(repos[0], WORK, ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache"))
prereg = hashlib.sha256(open(f"{WORK}/PREREGISTRATION.md", "rb").read()).hexdigest()
assert prereg == EXPECTED_PREREG_SHA256, f"PREREGISTRATION.md hash mismatch: {prereg}"
for f in ("audit/stage1_kaggle/stage1_report.json", "audit/stage1_kaggle/bank_frames_v1_2_1.csv"):
    assert os.path.exists(f"{WORK}/{f}"), f"{f} missing: upload the repository at the Stage 2 commit"
print("repository:", repos[0], "->", WORK, "| PREREGISTRATION.md hash OK | Stage 1 record present")

In [ ]:
# 2. Pinned packages into an isolated folder; torch, torchvision and timm come from the Kaggle image
DEPS = "/tmp/stage2_pydeps"
shutil.rmtree(DEPS, ignore_errors=True)
for args in (["-r", f"{WORK}/requirements.txt"], ["--no-deps", "-r", f"{WORK}/requirements-stage2.txt"]):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-warn-script-location",
                        "--target", DEPS, *args], capture_output=True, text=True)
    print(r.stdout[-1000:], r.stderr[-1500:])
    assert r.returncode == 0, "pip install failed: is Internet ON? (see the error above)"
ENV = dict(os.environ, PYTHONPATH=DEPS, PYTHONNOUSERSITE="1")
chk = subprocess.run([sys.executable, "-c",
    "import sys, numpy, torch, torchvision, timm, faiss, cv2; "
    "assert torch.cuda.is_available(), 'no GPU: set Accelerator to GPU'; "
    "torch.from_numpy(numpy.zeros(3)); "
    "print(sys.version.split()[0], numpy.__version__, torch.__version__, torchvision.__version__, timm.__version__, "
    "faiss.__version__, cv2.__version__, torch.version.cuda, torch.cuda.get_device_name(0).replace(' ', '_'))"],
    env=ENV, capture_output=True, text=True)
print("python numpy torch torchvision timm faiss cv2 cuda gpu:", chk.stdout.strip(), chk.stderr[-1500:])
assert chk.returncode == 0 and chk.stdout.split()[1] == "2.4.4", "environment check failed (see above)"

In [ ]:
# 3. Dataset root (the folder with CSV/ and IMAGE/); only folder names are listed here
roots = []
for base, dirs, files in os.walk("/kaggle/input"):
    if "CSV" in dirs and "IMAGE" in dirs:
        roots.append(base)
    dirs[:] = [d for d in dirs if d not in SKIP_DIRS]
assert len(roots) == 1, f"expected exactly one TSFabrics root, found: {roots}"
ROOT = roots[0]
print("dataset root:", ROOT)

# 4. Scratch space for one fold of bank features (about 10 GB, deleted after its coresets are built)
cands = [p for p in ("/kaggle/tmp", "/tmp") if os.path.isdir(p)]
free = {p: shutil.disk_usage(p).free / 1e9 for p in cands}
print("free GB:", {k: round(v, 1) for k, v in free.items()}, "| /kaggle/working:",
      round(shutil.disk_usage("/kaggle/working").free / 1e9, 1))
SCRATCH = max(free, key=free.get)
assert free[SCRATCH] > 12, "need more than 12 GB of scratch space for the bank features"
SCRATCH = os.path.join(SCRATCH, "stage2_work")
WORKERS = max(1, (os.cpu_count() or 2))
print("scratch:", SCRATCH, "| motion workers:", WORKERS)

In [ ]:
# 5. Smoke run (a few minutes): capped sizes, never the record. Its job is to fail fast on environment problems.
SMOKE = "/kaggle/working/stage2_smoke"
shutil.rmtree(SMOKE, ignore_errors=True)
with open("/kaggle/working/stage2_smoke_log.txt", "w") as log:
    r = subprocess.run([sys.executable, "scripts/run_stage2.py", "--dataset-root", ROOT, "--out", SMOKE,
                        "--work", SCRATCH, "--workers", str(WORKERS), "--smoke"],
                       cwd=WORK, env=ENV, stdout=log, stderr=subprocess.STDOUT, text=True)
print("smoke exit code:", r.returncode)
print(open("/kaggle/working/stage2_smoke_log.txt").read()[-2500:])
assert os.path.exists(f"{SMOKE}/stage2_report.json"), "the smoke run did not complete: see the log above"

In [ ]:
# 6. The Stage 2 run of record. About 1 to 2 hours. Resumable: rerunning this cell skips finished phases.
OUT = "/kaggle/working/stage2_out"
with open("/kaggle/working/stage2_log.txt", "a") as log:
    r = subprocess.run([sys.executable, "scripts/run_stage2.py", "--dataset-root", ROOT, "--out", OUT,
                        "--work", SCRATCH, "--workers", str(WORKERS)],
                       cwd=WORK, env=ENV, stdout=log, stderr=subprocess.STDOUT, text=True)
print("exit code:", r.returncode)
lines = [l for l in open("/kaggle/working/stage2_log.txt").read().splitlines() if "Subsampling" not in l]
print("\n".join(lines[-40:]))

In [ ]:
# 7. Summary and the small record files to commit (coreset vectors and validation maps stay in stage2_out/)
rep = json.load(open(f"{OUT}/stage2_report.json"))
print("STAGE 2 RESULT:", rep["result"])
for k, v in sorted(rep["acceptance"].items()):
    print(f"  {k:<14} {'PASS' if v else ('not applicable' if v is None else 'FAIL')}")
ph = rep["phases"]
if "A" in ph:
    for f, x in ph["A"]["folds"].items():
        t = x["tau_r"]
        print(f"tau_r {f}: {t['tau_r']}  ({t.get('pairs_at_tau')} of {t['pairs']} pairs at tau, "
              f"{t.get('consistent_at_tau')} consistent) | validation statuses {x['validation_status_counts']}")
        a = x["AT-06b"]
        print(f"  AT-06b {f}: rule={a.get('rule')} frames={a.get('frames')} holding={a.get('holding')}/200")
    print("frames without a motion estimate:", {k: v for k, v in ph["A"]["frames_without_estimate"].items() if v})
if "B" in ph:
    print("AT-02 max rel:", ph["B"]["AT-02"]["max_rel"])
if "C" in ph:
    print("seed 0 rebuilt identically:", ph["C"]["seed0_rebuild_identical"])
if "D" in ph:
    for f, x in ph["D"]["AT-03"].items():
        print(f"AT-03 {f}: err {x['max_abs_err_over_max_d64']:.3g} (tol 1e-4) argmin agreement {x['argmin_agreement']}")
    print("AT-04a:", ph["D"]["AT-04a"]["bit_identical"])
    print("AT-04b max rel:", max(ph["D"]["AT-04b"]["rel_maps"].values()))
print("read violations:", {n: p.get("read_violations") for n, p in ph.items() if p.get("read_violations")} or "none")
e = rep["environment"]
print("python", e["python"].split()[0], "| numpy", e["numpy"], "| torch", e["torch"], "| torchvision",
      e["torchvision"], "| cuda", e.get("cuda"), "| gpu", e["gpu"].get("nvidia_smi"))
print("files fingerprint:", rep["files_fingerprint"])
rp = f"{OUT}/stage2_report.json"
print("report sha256:", hashlib.sha256(open(rp, "rb").read()).hexdigest())

keep = ["stage2_report.json", "frozen_manifest.json", "stage2_unit_tests.xml"] + \
       [f"phase_{n}.json" for n in ("0", "A", "B", "C", "D", "E")]
with zipfile.ZipFile("/kaggle/working/stage2_record.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for n in keep:
        if os.path.exists(f"{OUT}/{n}"):
            z.write(f"{OUT}/{n}", n)
    for n in sorted(os.listdir(f"{OUT}/frozen")):
        if n.endswith((".json", ".lock")) or n.endswith("_indices.npy"):
            z.write(f"{OUT}/frozen/{n}", f"frozen/{n}")
    z.write("/kaggle/working/stage2_log.txt", "stage2_log.txt")
print("record zip:", "/kaggle/working/stage2_record.zip",
      hashlib.sha256(open("/kaggle/working/stage2_record.zip", "rb").read()).hexdigest())